In [1]:
from help import *

- load all the historical data and universe

In [2]:
star_board = pd.read_csv("star_board.csv")
tickers = star_board["stock_code"].tolist()

In [3]:
def load_history(ticker):
	tik = ticker.split(".")[0]
	t = pd.read_csv(f"ohlcv/{ticker}")
	t['ticker'] = tik
	return t
history = pd.concat([load_history(f) for f in os.listdir("ohlcv") if f.endswith(".csv")])
history["date"] = pd.to_datetime(history["date"]).dt.date
history = history.sort_values(["ticker", "date"])

- rebal

In [4]:
eff_date = dt.date(2026, 6, 12)
annc_date = dt.date(2026, 5, 30)
hist_start = dt.date(2025, 5, 1)
cutoff_date = dt.date(2026, 4, 30)
cutoff_10 = (cutoff_date + pd.offsets.BDay(10)).date()

- get listing date

In [5]:
tu = pd.DataFrame(history.groupby('ticker')['date'].min()).reset_index()
tu.columns = ['ticker', 'listing_date']
tu = tu.sort_values('listing_date').reset_index(drop=True)

In [6]:
info_l = []
for tik in tickers:
	tik_i = json.load(open(f"sse_info/{tik}.json", "r"))
	shs_titak = tik_i.get("sharesOutstanding", 0)
	shs_float = tik_i.get("floatShares", 0)
	exch = tik_i["exchange"]
	sector = tik_i.get("sector", "")

	d = {
		"ticker": tik,
		"shs_os_yf": shs_titak,
		"ff_shs_yf": shs_float,
		"exchange": exch,
		"sector": sector
	}
	info_l.append(d)

# create dataframe from info_l
info_df = pd.DataFrame(info_l)
info_df['ticker'] = info_df['ticker'].astype(str)

- add scraped data from sse

In [7]:
t_sse = pd.read_csv('sse.csv')
t_sse['ticker'] = t_sse['ticker'].astype(str)

# merge info_df and tu on ticker
tu = pd.merge(info_df, tu, on="ticker", how="outer")
tu = pd.merge(tu, t_sse, on="ticker", how="outer")
tu['shs_yf_to_sse_ratio'] = tu['shs_os_yf'] / tu['shares_total']

- special treatment securities

In [8]:
st_securities = tu[tu['special_treatment']]['ticker'].tolist()
qt.log.info(f"[{len(st_securities)}] ST securities: {st_securities}")

[JUSTY.LOG]	2026-05-09 13:36:32,982 - qt.common.help - INFO - [12] ST securities: ['688022', '688033', '688053', '688066', '688076', '688184', '688201', '688270', '688287', '688496', '688622', '688646']


- eligibility
	- Listing time > 6 months. 
		1. If no of securities listed > 12 months is b/w 100 to 150 then requirement changes to > 12 months
	- For securities with daily avg total market_cap since initial listing in top 5, listing time should be > 3 months as of 10th trading days after end date of data (cutoff date)
	- For securities with daily avg total market_cap since initial listing in top 3, listint time should be > 1 month
	- Non-* ST securities
	- No violation of laws/reg, no financial problems etc

In [9]:
history_after_cof = history[
	(history['date'] <= cutoff_date) &
	(history['date'] >= hist_start)
	].sort_values(['ticker', 'date']).reset_index(drop=True)

In [10]:
sse_holidays_t = pd.read_csv("sse_holidays.csv")
sse_holidays_t['date'] = pd.to_datetime(sse_holidays_t['date']).dt.date
sse_holidays = set(sse_holidays_t['date'].tolist())

/tmp/ipykernel_420328/1369734096.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sse_holidays_t['date'] = pd.to_datetime(sse_holidays_t['date']).dt.date


In [11]:
univ = tu[['ticker', 'listing_date', 'name', 'market_cap', 'special_treatment', 'shares_total']].copy()
univ = univ[~univ['special_treatment']].reset_index(drop=True)
univ['month_to_cof'] = univ['listing_date'].apply(lambda d: (cutoff_date.year - d.year) * 12 + (cutoff_date.month - d.month)  - int(cutoff_date.day < d.day))
univ['month_to_cof10'] = univ['listing_date'].apply(lambda d: (cutoff_10.year - d.year) * 12 + (cutoff_10.month - d.month)  - int(cutoff_10.day < d.day))

In [12]:
no_of_securities_mt_12m = len(univ[univ['month_to_cof'] >= 12])
if no_of_securities_mt_12m > 100:
	listing_month_cutoff = 12
else:
	listing_month_cutoff = 6
qt.log.info(f"Listing month cutoff: {listing_month_cutoff} months (securities with month_to_cof >= {listing_month_cutoff}: {no_of_securities_mt_12m})")

[JUSTY.LOG]	2026-05-09 13:36:34,897 - qt.common.help - INFO - Listing month cutoff: 12 months (securities with month_to_cof >= 12: 558)


In [ ]:
history_after_cof = pd.merge(history_after_cof, univ[['ticker', 'shares_total']], on='ticker', how='left')
history_after_cof['total_mcap'] = history_after_cof['close'] * history_after_cof['shares_total']
history_after_cof['val_traded'] = history_after_cof['close'] * history_after_cof['volume']
history_after_cof = history_after_cof[history_after_cof['ticker'].isin(univ['ticker'])].reset_index(drop=True)

# avg total market cap and avg value traded for each ticker
avg_tmcap = history_after_cof.groupby('ticker').agg({
	'total_mcap': 'mean',
	'val_traded': 'mean'
}).reset_index().sort_values('total_mcap', ascending=False).reset_index(drop=True)
avg_tmcap.columns = ['ticker', 'avg_total_mcap', 'avg_val_traded']

In [15]:
univ = pd.merge(univ, avg_tmcap, on='ticker', how='left')
univ['tmcap_rank'] = univ['avg_total_mcap'].rank(ascending=False, method='min')
univ['vtrad_rank'] = univ['avg_val_traded'].rank(ascending=False, method='min')

In [ ]:
# read weight of existing index
curr_s50 = pd.read_csv('star50_etf.csv')
curr_s50['ticker'] = curr_s50['ticker'].astype(str)
curr_s50['curr_weight'] = curr_s50['weight']

# add weight column
univ = pd.merge(univ, curr_s50[['ticker', 'curr_weight']], on='ticker', how='left')

- add eligibility

In [19]:
univ['listing_elig'] = (
	((univ['tmcap_rank'] <= 3) & (univ['month_to_cof'] >= 1)) |
	((univ['tmcap_rank'] <= 5) & (univ['month_to_cof10'] >= 3)) |
	(univ['month_to_cof'] >= listing_month_cutoff)
)

In [24]:
univ = univ[univ['listing_elig']].reset_index(drop=True)
univ['vtrad_rank2'] = univ['avg_val_traded'].rank(ascending=False, method='min')

In [29]:
qt.view(univ[univ['vtrad_rank2'] > 0.9*len(univ)])

Grid(columns_fit='auto', compress_data=True, css_rules_down=['.number-cell {text-align: left;width: 10;}', '.l…

In [38]:
univ = univ[univ['vtrad_rank2'] <= 0.9*len(univ)].reset_index(drop=True)

In [39]:
univ['tmcap_rank2'] = univ['avg_total_mcap'].rank(ascending=False, method='min').reset_index(drop=True)

In [40]:
qt.view(univ)

Grid(columns_fit='auto', compress_data=True, css_rules_down=['.number-cell {text-align: left;width: 10;}', '.l…

In [41]:
univ[univ['ticker'].isin(
	['688498', '688110', '688002']
)]

,ticker,listing_date,name,market_cap,special_treatment,shares_total,month_to_cof,month_to_cof10,avg_total_mcap,avg_val_traded,tmcap_rank,vtrad_rank,curr_weight,listing_elig,vtrad_rank2,tmcap_rank2
1,688002,2019-07-22,睿创微纳,"6,858,441.28",False,"465,736,879.00",81,81,"39,947,685,461.02","615,029,177.95",52.00,88.00,NaN,True,75.00,41.00
63,688110,2021-12-10,东芯股份,"6,847,353.00",False,"442,249,758.00",52,53,"41,269,522,968.57","2,254,118,260.78",50.00,12.00,NaN,True,10.00,39.00
268,688498,2022-12-21,源杰科技,"13,536,766.85",False,"85,947,726.00",40,40,"46,586,620,045.09","2,104,424,545.55",41.00,13.00,NaN,True,11.00,31.00
